In [ ]:
# set library
import json
from transformers import AutoTokenizer
from tqdm import tqdm
import re
import numpy as np
from collections import defaultdict
from scipy import integrate
import scipy
import math

In [ ]:
# Set Variables
model_flops = {
    'meta-llama/Llama-3.2-3B-Instruct': 3000000000,
    'Qwen/Qwen2.5-3B-Instruct': 3000000000,
    'google/gemma-3-4b-it': 4000000000,
    'Qwen/Qwen2.5-7B-Instruct': 7000000000,
    'google/gemma-3-27b-it': 27000000000,
}

# dataset="gsm8k"
dataset = "math"
# model = 'meta-llama/Llama-3.2-3B-Instruct'
model = "Qwen/Qwen2.5-7B-Instruct"
# model = 'google/gemma-3-4b-it'

cp_threshold = 0.90
beta_threshold = 0.95

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code=True)
if dataset == 'omnimath':
    with open(f"./logs/self_certainty/sc_16_{dataset}_2048_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        data = [json.loads(line) for line in f]
else:
    with open(f"./logs/self_certainty/sc_16_{dataset}_1024_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

In [ ]:
def apply_chat_template(dataset, question, response):
    if dataset == "gpqa_diamond" or dataset == "arcChallenge":
        prompt = f"{question}\n\nBased on the above, what is the single, most likely answer choice? Answer in the format \"The correct answer is (insert answer here)\"."
        chat_template = [{'role': 'user', 'content': prompt}, {"role": "assistant", "content": response}]
    else:
        chat_template = [{'role': 'user', 'content': question}, {"role": "assistant", "content": response}]

    return tokenizer.apply_chat_template(chat_template, tokenize=False, add_generation_prompt=False)

def extract_boxed_content(text):
    start = text.find(r"\boxed{")
    if start == -1:
        return None
    i = start + len(r"\boxed{")
    depth = 1
    content = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        content.append(c)
        i += 1
    return "".join(content)

def clean(value):
    if dataset == 'gsm8k':
        value = value.replace(',','')
        numbers = re.findall(r"\d+(?:\.\d+)?", value)
        if len(numbers) > 0:
            value = numbers[-1]
        else:
            value= None
    else:
        final_value = extract_boxed_content(value)
        if final_value is None:
            if dataset == 'gpqa_diamond' or dataset == 'arcChallenge':
                value = re.findall(r"A|B|C|D|a|b|c|d", value)
                value = value[0].lower() if value else ''
            else:
                value = ""
        else:
            value = final_value.replace(' ','')
            if dataset == 'gpqa_diamond' or dataset == 'arcChallenge':
                value = value.lower()
    return value

In [ ]:
# aggregate results by question
results = defaultdict(list)
error_cnt=0
for inst in data:
    question = inst['question']


    if dataset == 'gsm8k':
        pred = inst['pred'].replace(' ','').strip()
        verdict = clean(inst['pred']) == clean(inst['answer'])
    else:
        pred = clean(inst['response'])
        if pred == '':
            pred = clean(inst['pred'])
            if pred == '':
                pred = inst['pred'].replace(' ','').strip()
        
        if pred == '':
            error_cnt += 1
        
        verdict = inst['verdict']

    response = inst['response']
    if pred == '':
        error_cnt+=1
    internal_value = None
    
    if question not in results:
        results[question] = []
    
    if dataset == 'gsm8k':
        results[question].append((clean(pred), internal_value, verdict, apply_chat_template(dataset, question, response)))
    else:
        results[question].append((pred, internal_value, verdict, apply_chat_template(dataset, question, response)))

# ESC results

In [ ]:
# check esc
from collections import Counter
import numpy as np
# Into a whole evaluation loop

window_size=4
correct_esc = 0
total_esc = 0
sample_size_list = []
total_response_length = 0

for question in results:
    sample = results[question]
    predictions = [inst[0] for inst in sample]
    # predVerdict = {inst[0]: inst[2] for inst in sample}
    predVerdict = {}
    # for inst in sample:
    #     if inst[0] not in predVerdict:
    #         predVerdict[inst[0]] = inst[2]
    #     else:
    #         # If any instance is correct, mark as correct
    #         predVerdict[inst[0]] = predVerdict[inst[0]] or inst[2]
    for inst in sample:
        if inst[0] not in predVerdict:
            predVerdict[inst[0]] = {}
        predVerdict[inst[0]][inst[2]] = predVerdict[inst[0]].get(inst[2], 0) + 1
    predictionCount = {pred: predictions.count(pred) for pred in predictions}
    # print(predictionCount)
    responses = [inst[3] for inst in sample]

    new_predictions = []
    for i in range(0, len(sample), window_size):
        # print(i, i+window_size)
        window = predictions[i:i+window_size]
        new_predictions.extend(window)
        if len(list(set(window))) == 1:
            break
    esc_prediction = Counter(new_predictions).most_common(1)[0][0]
    esc_verdict = predVerdict[esc_prediction].get(True, 0) >= predVerdict[esc_prediction].get(False, 0)
    consumed_responses = responses[:len(new_predictions)]
    sample_size_list.append(len(new_predictions))
    if esc_verdict:
        correct_esc += 1
    total_esc += 1
    total_response_length += sum(len(tokenizer.encode(resp, add_special_tokens=False)) for resp in consumed_responses)


correct_esc, total_esc, correct_esc/total_esc * 100, np.mean(sample_size_list), total_response_length/len(sample_size_list), total_response_length
print(f"====================ESC (window_size: {window_size})====================")
print("ESC Results:")
print("Correct:", correct_esc)
print("Total:", total_esc)
print("Accuracy: {:.2f}%".format(correct_esc/total_esc * 100))
print("Average Sample Size:", np.mean(sample_size_list))
print("Average Response Length per Question:", total_response_length/len(sample_size_list))
print("Total Response Length:", total_response_length)
print("Flops: ", total_response_length * model_flops[model], "FLOPS")
print("Average TFLOPS:", (total_response_length * model_flops[model]) / (len(sample_size_list) * 1e12))
print("==========================================")